# Development aggregate ClearML publication

Publishes only the sealed, allowlisted development tables and figures. It does not rerun evaluation or inference. Credentials are read from Kaggle Secrets and are never printed or retained.


In [1]:
from pathlib import Path
import hashlib
import importlib.util
import json
import os
import subprocess
import sys

WORK = Path('/kaggle/working/development-clearml-publication')
WORK.mkdir(parents=True, exist_ok=True)
RECEIPT = WORK / 'publication_receipt.json'
SUCCESS = WORK / 'publication_complete.json'
MOUNT = Path('/kaggle/input')


def candidates(name):
    found = []
    for path in MOUNT.rglob(name):
        if len(found) >= 20:
            raise ValueError(f'Too many {name} candidates; narrow the mounted resources')
        if path.is_file() and not path.is_symlink():
            found.append(path)
    return sorted(set(p.resolve() for p in found))


def unique(name, predicate):
    found = [path for path in candidates(name) if predicate(path)]
    if len(found) != 1:
        raise ValueError(f'Expected one {name} for this publication; found {len(found)}')
    return found[0]


# Resolve only markers from the three explicitly attached private resources.
manifest_path = unique('prepared_source_manifest.json', lambda p: (p.parent / 'scripts/publish_next_stage.py').is_file())
code_root = manifest_path.parent
manifest = json.loads(manifest_path.read_text())
for relative in ('scripts/current_notebook_runtime.py', 'scripts/publish_next_stage.py', 'scripts/publish_clearml.py'):
    path = code_root / relative
    expected = manifest.get('files', {}).get(relative)
    if not expected or hashlib.sha256(path.read_bytes()).hexdigest() != expected:
        raise ValueError(f'Prepared source checksum differs: {relative}')

evaluation_lock = unique('evaluation_lock.json', lambda p: p.parent.name == 'development-evaluate')
report_lock = unique('report_lock.json', lambda p: p.parent.name == 'development-evaluate-report')
evaluation_root = evaluation_lock.parent
report_root = report_lock.parent
tracking_profile = unique('runtime_profile_lock.json', lambda p: 'setup-tracking' in p.as_posix())
tracking_wheelhouse = unique('wheelhouse_manifest.json', lambda p: 'setup-tracking' in p.as_posix())

os.environ.update(
    VPI_TRACKING_PROFILE_ROOT=str(tracking_profile.parent),
    VPI_TRACKING_WHEELHOUSE=str(tracking_wheelhouse.parent),
    VPI_TRACKING_TARGET='/tmp/vpi-runtime-tracking-development-publish',
    CLEARML_API_HOST='https://api.clear.ml',
    CLEARML_WEB_HOST='https://app.clear.ml',
    CLEARML_FILES_HOST='https://files.clear.ml',
    CLEARML_LOG_ENVIRONMENT='',
)

sys.path.insert(0, str(code_root / 'scripts'))
# Kaggle's notebook worker may pre-import NumPy before the first cell. Install
# the sealed target in a clean child interpreter, then validate and use that
# completed target from this notebook process.
installer = '\n'.join([
    'import json, sys',
    f"sys.path.insert(0, {str(code_root / 'scripts')!r})",
    'from current_notebook_runtime import configure_tracking_runtime',
    f"runtime = configure_tracking_runtime({str(code_root)!r}, allow_install=True)",
    "print(json.dumps({'status': runtime['status']}))",
])
install_result = subprocess.run(
    [sys.executable, '-B', '-c', installer],
    timeout=320,
    check=False,
    capture_output=True,
    text=True,
    env=dict(os.environ),
)
if install_result.returncode != 0:
    raise RuntimeError('Isolated tracking runtime installation failed')

from current_notebook_runtime import configure_tracking_runtime
runtime = configure_tracking_runtime(code_root, allow_install=False)

# Kaggle's secret client imports scientific dependencies. Install the sealed
# tracking runtime first so the current interpreter has not imported them yet.
# Credentials are then read without displaying or writing their values.
from kaggle_secrets import UserSecretsClient
client = UserSecretsClient()
for key in ('CLEARML_API_ACCESS_KEY', 'CLEARML_API_SECRET_KEY'):
    try:
        value = client.get_secret(key)
    except Exception as error:
        raise RuntimeError(f'Kaggle Secret {key} is missing or not enabled for this notebook') from error
    if not value:
        raise RuntimeError(f'Kaggle Secret {key} is empty')
    os.environ[key] = value

environment = dict(os.environ)
environment['PYTHONPATH'] = os.pathsep.join(filter(None, [
    os.environ['VPI_TRACKING_TARGET'], str(code_root / 'src'), str(code_root / 'scripts'),
    os.environ.get('PYTHONPATH')]))
command = [
    sys.executable, '-B', str(code_root / 'scripts/publish_next_stage.py'),
    '--stage', 'dev-evaluate', '--evaluation-root', str(evaluation_root),
    '--report-root', str(report_root), '--receipt', str(RECEIPT),
]
result = subprocess.run(command, timeout=240, check=False, capture_output=True, text=True, env=environment)
record = json.loads(RECEIPT.read_text()) if RECEIPT.is_file() else {}
if result.returncode != 0 or record.get('status') != 'completed':
    raise RuntimeError(f"Aggregate publication did not complete: {record.get('error_type', 'missing receipt')}")

safe = {key: record[key] for key in (
    'status', 'task_id', 'task_url', 'binding_sha256', 'scientific_lock_hashes',
    'artifact_hashes', 'remote_artifacts_verified') if key in record}
SUCCESS.write_text(json.dumps(safe, indent=2, sort_keys=True) + '\n')
print({
    'status': safe['status'],
    'task_id': safe.get('task_id'),
    'task_url': safe.get('task_url'),
    'published_artifact_count': len(safe.get('artifact_hashes', {})),
    'scientific_results_recomputed': False,
    'credentials_persisted': False,
    'tracking_runtime': runtime['status'],
})


{'status': 'completed', 'task_id': '12c05d6ba18142549b01d647b91345ba', 'task_url': 'https://app.clear.ml/projects/62a09a229f184cb7bbe50da59b4226e7/experiments/12c05d6ba18142549b01d647b91345ba/output/log', 'published_artifact_count': 16, 'scientific_results_recomputed': False, 'credentials_persisted': False, 'tracking_runtime': 'pinned_tracking_runtime_ready'}
